In [ ]:
# 確認 GPU 已連接
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.stdout else "No GPU found!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# Clone segmentation branch；如果已經存在就略過，方便重跑 cell
if [ ! -d /content/CV-Assignment2-Group6/.git ]; then
  git clone -b segmentation https://github.com/linenmin/CV-Assignment2-Group6.git /content/CV-Assignment2-Group6
else
  echo "Repo already exists, skip clone."
fi

# 建立資料集捷徑；如果已存在就略過
if [ ! -e /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026 ]; then
  ln -s /content/drive/MyDrive/kul-computer-vision-ga-2-2026 \
        /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026
else
  echo "Dataset link already exists, skip symlink."
fi

# 確認資料集連結成功
echo "Train images: $(ls /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026/train/img | wc -l)"
echo "Test images: $(ls /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026/test/img | wc -l)"

In [ ]:
%%bash
# 1. 安裝已知可以配合 Colab Python 3.12 的組合：PyTorch 2.4.0 + mmcv 2.2.0 + mmseg 1.2.2
pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q
pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html -q
pip install mmsegmentation==1.2.2 -q

# 2. 解除 mmsegmentation 對 mmcv 2.2.0 的版本限制
sed -i "s/MMCV_MAX = '2.2.0'/MMCV_MAX = '2.3.0'/g" /usr/local/lib/python3.12/dist-packages/mmseg/__init__.py

# 3. 修復 pkgutil / tokenizer 相關依賴
pip install -q --upgrade setuptools timm ftfy regex

# 4. 安裝專案本身
cd "/content/CV-Assignment2-Group6/Semantic segmentation"
pip install -q -e .

In [ ]:
import torch, mmcv, mmseg
print(f'PyTorch: {torch.__version__}')
print(f'mmcv: {mmcv.__version__}')
print(f'mmsegmentation: {mmseg.__version__}')
print('All OK! Ready to train.')

In [ ]:
%%bash
cat > "/content/CV-Assignment2-Group6/Semantic segmentation/configs/experiments/segnext_s_512x512_adamw_poly_v5_ce_dice.py" << 'EOF'
_base_ = ["./segnext_s_512x512_adamw_poly_v1.py"]

data_root = '/content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026'
project_root = '/content/CV-Assignment2-Group6/Semantic segmentation'

train_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        ann_file=f'{project_root}/data/splits/train.txt',
        data_prefix=dict(img_path='train/img', seg_map_path='train/seg')
    )
)
val_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        ann_file=f'{project_root}/data/splits/val.txt',
        data_prefix=dict(img_path='train/img', seg_map_path='train/seg')
    )
)
test_dataloader = val_dataloader

# V5: keep V1 architecture/training setup, only change decode_head loss.
# CE keeps stable pixel classification; Dice encourages mask overlap and foreground balance.
model = dict(
    decode_head=dict(
        loss_decode=[
            dict(type='CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0),
            dict(type='DiceLoss', use_sigmoid=False, activate=True, loss_weight=0.5, eps=1e-5),
        ]
    )
)

work_dir = "./outputs/logs/exp_v5_ce_dice"
EOF
echo "V5 CE+Dice config created!"

In [ ]:
%%bash
cd "/content/CV-Assignment2-Group6/Semantic segmentation"
python scripts/train.py \
    --config configs/experiments/segnext_s_512x512_adamw_poly_v5_ce_dice.py \
    --num-workers 4 --batch-size 2 \
    --work-dir outputs/logs/exp_v5_ce_dice

In [ ]:
%%bash
cd "/content/CV-Assignment2-Group6/Semantic segmentation"

# 找到最佳 checkpoint（自動取檔名）
BEST_CKPT=$(ls outputs/logs/exp_v5_ce_dice/best_mIoU_*.pth 2>/dev/null | head -1)
echo "Best checkpoint: $BEST_CKPT"
if [ -z "$BEST_CKPT" ]; then
  echo "No best checkpoint found. Stop before prediction."
  exit 1
fi

# 產生預測遮罩
python scripts/predict_test_segmentation.py \
    --config configs/experiments/segnext_s_512x512_adamw_poly_v5_ce_dice.py \
    --checkpoint $BEST_CKPT \
    --output-dir outputs/predictions/exp_v5_ce_dice_test

# 修正新版 pandas 對 tuple 欄位索引的相容性問題
python - <<'PY'
from pathlib import Path
p = Path('src/ga2_seg/submission.py')
text = p.read_text()
text = text.replace('df.loc[idx, CLASS_NAMES]', 'df.loc[idx, list(CLASS_NAMES)]')
p.write_text(text)
print('submission.py pandas compatibility patch applied')
PY

# 匯出 submission.csv
python scripts/export_submission.py \
    --prediction-dir outputs/predictions/exp_v5_ce_dice_test \
    --output-path outputs/submissions/submission_exp_v5_ce_dice.csv \
    --classification-fill 0

# 產生訓練分析圖表
python scripts/analyze_training_run.py \
    --work-dir outputs/logs/exp_v5_ce_dice

In [ ]:
%%bash
mkdir -p /content/drive/MyDrive/CV_Assignment_Outputs/v5_ce_dice
cp -r "/content/CV-Assignment2-Group6/Semantic segmentation/outputs" \
      /content/drive/MyDrive/CV_Assignment_Outputs/v5_ce_dice/
echo "Backup complete!"